# Copula-GARCH

In [61]:
import numpy as np
import pandas as pd

from arch import arch_model 

from copulae import pseudo_obs # function to get pseudo-observations for copula fitting
from copulae import GaussianCopula

import yfinance as yf

## Functions

Fitting GARCH models

In [2]:
def fit_garch(returns):
    """
    Fit a ARMA(1,1)-GARCH(1,1) model to the returns and return the fitted model.
    """
    model = arch_model(returns*100, vol="Garch", p=1, q=1, dist="t") # scale returns for better convergence

    res = model.fit(disp="off", show_warning=False) # disp="off" to suppress output, show_warning=False to ignore convergence warnings

    return res

## 1. Basics

### 1.1 Load data

In [3]:
TICKERS = ["NVDA", "AAPL", "WMT", "LLY", "JPM"]

prices = yf.download(TICKERS, start="2020-01-01", end="2024-06-30")["Close"]

returns = np.log(prices / prices.shift(1)).dropna() # log returns

returns.head()

[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,JPM,LLY,NVDA,WMT
Date,,,,,
2020-01-03,-0.009769,-0.013285,-0.003334,-0.016135,-0.008867
2020-01-06,0.007936,-0.000795,0.003712,0.004185,-0.002038
2020-01-07,-0.004714,-0.017147,0.001888,0.012034,-0.009308
2020-01-08,0.015959,0.007771,0.009015,0.001874,-0.003438
2020-01-09,0.021018,0.003645,0.016393,0.010923,0.010278


### 1.2 Setup

In [4]:
ROLL_WIND = 250
FORECAST = 100
N_NIM = 10000

ALPHA = 0.05

SEED = 42

weights = np.ones(len(TICKERS)) / len(TICKERS)

## 2. Rolling window

## SANDBOX with R implementation

### Load return data

In [23]:
returns_aapl = returns["AAPL"]
returns_jpm = returns["JPM"]

### Using R's rugarch package in python

In [6]:
# rpy2 
import rpy2

import rpy2.robjects as ro
from rpy2.robjects.packages import importr # importr is used to import R packages


In [13]:
R --version

NameError: name 'R' is not defined

In [10]:
# loading R packages
utils = importr('utils') # utils is used to install R packages

copula = importr("copula")
rugarch = importr("rugarch")

In [11]:
# check versions of R packages
rugarch.__version__

'1.5-5'

In [12]:
# check versions of R packages
copula.__version__

'1.1-7'

#### Fitting the GARCH model

In [14]:
# GARCH(1,1) specification in R
spec = rugarch.ugarchspec(variance_model = ro.ListVector({'model': "sGARCH", 'garchOrder': ro.IntVector([1, 1])}), # specify GARCH(1,1) model
                          mean_model = ro.ListVector({'armaOrder': ro.IntVector([1, 1]), 'include.mean': True}), # specify ARMA(1,1) model for the mean
                          distribution_model = "std") # specify Student's t distribution for the innovations

In [24]:
# fit the model to the returns (scale returns for better convergence)
fit_aapl = rugarch.ugarchfit(spec, ro.FloatVector(returns_aapl.values * 100)) # fit the model to the returns (scale returns for better convergence)
fit_jpm = rugarch.ugarchfit(spec, ro.FloatVector(returns_jpm.values * 100)) # fit the model to the returns (scale returns for better convergence)

#### Extracting standardized residuals

In [25]:
residuals_aapl = np.array(rugarch.residuals(fit_aapl)) # Get the residuals from the fitted model
sigma_aapl = np.array(rugarch.sigma(fit_aapl)) # Get the conditional volatility from the fitted model
residuals_standardized_aapl = residuals_aapl / sigma_aapl # Standardize the residuals

residuals_jpm = np.array(rugarch.residuals(fit_jpm))
sigma_jpm = np.array(rugarch.sigma(fit_jpm))
residuals_standardized_jpm = residuals_jpm / sigma_jpm

#### Forecast values

In [26]:
# One-step-ahead forecast for mean and conditional volatility
forecast_aapl = rugarch.ugarchforecast(fit_aapl, n_ahead=1)
forecast_jpm = rugarch.ugarchforecast(fit_jpm, n_ahead=1)

In [27]:
# Mean forecast for the next day (scale back the mean forecast)
mu_forecast_aapl = np.array(rugarch.fitted(forecast_aapl))[0] / 100 # scale back the mean forecast
mu_forecast_jpm = np.array(rugarch.fitted(forecast_jpm))[0] / 100 # scale back the mean forecast

# Volatility forecast for the next day (scale back the volatility forecast)
sigma_forecast_aapl = np.array(rugarch.sigma(forecast_aapl))[0] / 100 # scale back the volatility forecast
sigma_forecast_jpm = np.array(rugarch.sigma(forecast_jpm))[0] / 100 # scale back the volatility forecast

In [29]:
mu_forecast_aapl, sigma_forecast_aapl, mu_forecast_jpm, sigma_forecast_jpm

(array([0.00147206]),
 array([0.01847455]),
 array([0.00103727]),
 array([0.01276176]))

In [35]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

matrix = np.array([a, b])
matrix

array([[1, 2, 3],
       [4, 5, 6]])

#### Pseudo observations

In [40]:
residuals_standardized_aapl

array([[-0.5360317 ],
       [ 0.32408252],
       [-0.30942911],
       ...,
       [ 0.95810541],
       [ 0.16134835],
       [-0.93760526]], shape=(1129, 1))

In [ ]:
# Speichern der standardisierten Residuen in der Matrix
residuals_standardized_matrix = np.column_stack((residuals_standardized_aapl, residuals_standardized_jpm))
residuals_standardized_matrix2 = np.array((residuals_standardized_aapl, residuals_standardized_jpm)) # rausnhemen
residuals_standardized_matrix, residuals_standardized_matrix2

(array([[-0.5360317 , -0.69340181],
        [ 0.32408252, -0.10065144],
        [-0.30942911, -0.96328314],
        ...,
        [ 0.95810541, -0.34285771],
        [ 0.16134835,  0.61168582],
        [-0.93760526,  1.1652201 ]], shape=(1129, 2)),
 array([[[-0.5360317 ],
         [ 0.32408252],
         [-0.30942911],
         ...,
         [ 0.95810541],
         [ 0.16134835],
         [-0.93760526]],
 
        [[-0.69340181],
         [-0.10065144],
         [-0.96328314],
         ...,
         [-0.34285771],
         [ 0.61168582],
         [ 1.1652201 ]]], shape=(2, 1129, 1)))

In [50]:
from copulae import pseudo_obs

In [51]:
u2 = pseudo_obs(residuals_standardized_matrix) # Get the pseudo-observations from the standardized residuals
u2

array([[0.27168142, 0.22212389],
       [0.65486726, 0.47079646],
       [0.36548673, 0.1460177 ],
       ...,
       [0.85221239, 0.35309735],
       [0.58230088, 0.76725664],
       [0.14159292, 0.91150442]], shape=(1129, 2))

In [ ]:
u = copula.pobs(ro.FloatVector(residuals_standardized_matrix)) # Get the pseudo-observations from the standardized residuals
u

NotImplementedError: Conversion 'py2rpy' not defined for objects of type '<class 'numpy.ndarray'>'

In [56]:
u2.shape[1]

2

#### Copulas

In [60]:
# Gaussian copula
cop_gaussian = copula.normalCopula(dim=u2.shape[1], dispstr="un")

fit_gaussian = copula.fitCopula(cop_gaussian, u2, method = "mpl") # Fit the Gaussian copula to the pseudo-observations

NotImplementedError: Conversion 'py2rpy' not defined for objects of type '<class 'numpy.ndarray'>'